# 04 — Complex Technical Specifications (Team 4)

## Columns Assigned
`Motor Hacmi`, `Motor Gücü`, `Ort. Yakıt Tüketimi`, `Ortalama Yakıt Tüketimi`,
`Şehir İçi Yakıt Tüketimi`, `Şehir Dışı Yakıt Tüketimi`, `Tork`,
`Maksimum Güç`, `Minimum Güç`, `Hızlanma (0-100)`, `Maksimum Hız`,
`Uzunluk`, `Genişlik`, `Yükseklik`, `Ağırlık`, `Boş Ağırlığı`,
`Aks Aralığı`, `Ön Lastik`

## What is Expected as Output
- `Motor Hacmi`: some values are ranges like "1401 - 1600 cm3" → take the midpoint (1500); others are "1461 cc" → strip "cc". Result: float
- `Motor Gücü`: same range logic as Motor Hacmi e.g. "101 - 125 HP" → midpoint (113); strip "hp". Result: float
- `Ort. Yakıt Tüketimi`, `Ortalama Yakıt Tüketimi`, `Şehir İçi Yakıt Tüketimi`, `Şehir Dışı Yakıt Tüketimi`: strip " lt", replace comma decimal separator with dot → float
- `Tork`: strip " nm" → float
- `Maksimum Güç`: strip " rpm" → float
- `Minimum Güç`: strip " rpm" → float
- `Hızlanma (0-100)`: strip " sn", replace comma with dot → float
- `Maksimum Hız`: strip " km/s" → float
- `Uzunluk`, `Genişlik`, `Yükseklik`, `Aks Aralığı`: strip " mm" → float
- `Ağırlık`, `Boş Ağırlığı`: strip " kg" → float
- `Ön Lastik`: extract rim diameter from tire size strings like "225/45 R19" using regex → integer column renamed to `Jant Boyutu`; drop original `Ön Lastik` column
- All nulls filled with column median after unit stripping
- No nulls remaining, all columns numeric, no string columns

## Output variable: df_team4

---

> ## ⚠️ READ THIS BEFORE YOU WRITE A SINGLE LINE OF CODE ⚠️
>
> ### The code below is a PLACEHOLDER — it is NOT the final solution.
>
> The cells in this notebook show **one possible way** to process the assigned columns. You are **not** required to follow this approach. You may completely replace it with your own method if you believe yours is better suited to the data.
>
> ### What IS required — without exception:
>
> **Every code cell must be accompanied by a markdown cell that explains:**
> 1. **What you did** — describe the operation in plain language.
> 2. **Why you did it** — justify the choice. Why did you think this specific approach would work? What problem does it solve? Why this method and not another?
>
> A notebook that only contains code — or markdown cells that just repeat the code in words — will **not** be accepted.
>
> **Not acceptable:** `## Step 2` or `# fill nulls`
>
> **Acceptable:** *"We imputed missing values in `Kilometre` with the median rather than the mean because the column is heavily right-skewed (a small number of very high-mileage listings would pull the mean upward, misrepresenting the majority of vehicles). The median is robust to those outliers and better reflects a typical listing."*
>
> Document every decision. If you tried an approach and abandoned it, explain why.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_URL = "https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/raw_dataset.csv"
df_full = pd.read_csv(RAW_URL)
print(f"Loaded dataset: {df_full.shape}")
df_full.head(3)

In [ ]:
MY_COLUMNS = [
    'Motor Hacmi', 'Motor Gücü',
    'Ort. Yakıt Tüketimi', 'Ortalama Yakıt Tüketimi',
    'Şehir İçi Yakıt Tüketimi', 'Şehir Dışı Yakıt Tüketimi',
    'Tork', 'Maksimum Güç', 'Minimum Güç',
    'Hızlanma (0-100)', 'Maksimum Hız',
    'Uzunluk', 'Genişlik', 'Yükseklik',
    'Ağırlık', 'Boş Ağırlığı', 'Aks Aralığı', 'Ön Lastik'
]
df = df_full[MY_COLUMNS].copy()

## Step 1 — Null Analysis

In [ ]:
# Print null counts and percentage for each column
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
print(pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct}))

In [ ]:
# Helper: parse a range string 'a - b <unit>' → midpoint float, or a single value → float
def parse_range_midpoint(series, strip_patterns):
    s = series.astype(str).str.strip()
    for pat in strip_patterns:
        s = s.str.replace(pat, '', regex=False, case=False).str.strip()
    # Replace comma decimal separators with dots
    s = s.str.replace(',', '.', regex=False)
    # Detect range pattern 'low - high'
    is_range = s.str.contains(r'^\d[\.\d]*\s*-\s*\d', regex=True, na=False)
    result = pd.to_numeric(s.where(~is_range), errors='coerce')
    # For range rows, extract low and high and take midpoint
    parts = s[is_range].str.extract(r'([\d\.]+)\s*-\s*([\d\.]+)')
    if not parts.empty:
        midpoints = (pd.to_numeric(parts[0], errors='coerce') +
                     pd.to_numeric(parts[1], errors='coerce')) / 2
        result[is_range] = midpoints
    return result

In [ ]:
# Clean Motor Hacmi: strip 'cm3' and 'cc', handle ranges → midpoint
df['Motor Hacmi'] = parse_range_midpoint(df['Motor Hacmi'], [' cm3', 'cm3', ' cc', 'cc'])
df['Motor Hacmi'] = df['Motor Hacmi'].fillna(df['Motor Hacmi'].median())
print(f"Motor Hacmi nulls: {df['Motor Hacmi'].isnull().sum()}")

In [ ]:
# Clean Motor Gücü: strip 'HP' and 'hp', handle ranges → midpoint
df['Motor Gücü'] = parse_range_midpoint(df['Motor Gücü'], [' HP', 'HP', ' hp', 'hp'])
df['Motor Gücü'] = df['Motor Gücü'].fillna(df['Motor Gücü'].median())
print(f"Motor Gücü nulls: {df['Motor Gücü'].isnull().sum()}")

In [ ]:
# Clean fuel consumption columns: strip ' lt', replace comma decimal separator
for col in ['Ort. Yakıt Tüketimi', 'Ortalama Yakıt Tüketimi',
            'Şehir İçi Yakıt Tüketimi', 'Şehir Dışı Yakıt Tüketimi']:
    df[col] = (
        df[col].astype(str)
        .str.replace(' lt', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean Tork: strip ' nm'
df['Tork'] = (
    df['Tork'].astype(str)
    .str.replace(' nm', '', regex=False, case=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Tork'] = pd.to_numeric(df['Tork'], errors='coerce')
df['Tork'] = df['Tork'].fillna(df['Tork'].median())
print(f"Tork nulls: {df['Tork'].isnull().sum()}")

In [ ]:
# Clean Maksimum Güç and Minimum Güç: strip ' rpm'
for col in ['Maksimum Güç', 'Minimum Güç']:
    df[col] = (
        df[col].astype(str)
        .str.replace(' rpm', '', regex=False, case=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean Hızlanma (0-100): strip ' sn', replace comma with dot
df['Hızlanma (0-100)'] = (
    df['Hızlanma (0-100)'].astype(str)
    .str.replace(' sn', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Hızlanma (0-100)'] = pd.to_numeric(df['Hızlanma (0-100)'], errors='coerce')
df['Hızlanma (0-100)'] = df['Hızlanma (0-100)'].fillna(df['Hızlanma (0-100)'].median())
print(f"Hızlanma (0-100) nulls: {df['Hızlanma (0-100)'].isnull().sum()}")

In [ ]:
# Clean Maksimum Hız: strip ' km/s'
df['Maksimum Hız'] = (
    df['Maksimum Hız'].astype(str)
    .str.replace(' km/s', '', regex=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Maksimum Hız'] = pd.to_numeric(df['Maksimum Hız'], errors='coerce')
df['Maksimum Hız'] = df['Maksimum Hız'].fillna(df['Maksimum Hız'].median())
print(f"Maksimum Hız nulls: {df['Maksimum Hız'].isnull().sum()}")

In [ ]:
# Clean dimension columns: strip ' mm'
for col in ['Uzunluk', 'Genişlik', 'Yükseklik', 'Aks Aralığı']:
    df[col] = (
        df[col].astype(str)
        .str.replace(' mm', '', regex=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean weight columns: strip ' kg'
for col in ['Ağırlık', 'Boş Ağırlığı']:
    df[col] = (
        df[col].astype(str)
        .str.replace(' kg', '', regex=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean Ön Lastik: extract rim diameter from tire size strings like '225/45 R19' → Jant Boyutu
df['Jant Boyutu'] = (
    df['Ön Lastik']
    .astype(str)
    .str.extract(r'R(\d+)')[0]
)
df['Jant Boyutu'] = pd.to_numeric(df['Jant Boyutu'], errors='coerce')
df['Jant Boyutu'] = df['Jant Boyutu'].fillna(df['Jant Boyutu'].median()).astype(int)
# Drop original Ön Lastik column
df.drop(columns=['Ön Lastik'], inplace=True)
print(f"Jant Boyutu nulls: {df['Jant Boyutu'].isnull().sum()}")

In [ ]:
df_team4 = df.copy()
assert df_team4.isnull().sum().sum() == 0, "Nulls remain!"
assert df_team4.select_dtypes(exclude='number').shape[1] == 0, "Non-numeric columns remain!"
print(f"✅ Team 4 output ready. Shape: {df_team4.shape}")
print(df_team4.dtypes)